# 🔄 Data Orchestration: Setting up a Parent Pipeline

## What is Data Orchestration?
In a production environment, you do not run individual pipelines manually. Orchestration is the process of aggregating multiple standalone pipelines (e.g., On-Prem ingestion, API ingestion, and SQL incremental ingestion) into a single master workflow. This allows you to schedule them all at once and set up centralized monitoring and alerting.

## ➕ Step 1: Create the Parent Pipeline
To schedule multiple pipelines together, we must group them under one roof.
1. Go to your **Author** tab and click **New pipeline**.
2. Rename the pipeline to `parent pipeline`.

## ➕ Step 2: Use the 'Execute Pipeline' Activity
The core activity for orchestration in ADF is the **Execute Pipeline** activity, which simply triggers another pipeline to run.
1. Drag the **Execute Pipeline** activity onto the canvas.
2. Rename the activity (e.g., `execute onpre`).
3. Go to the **Settings** tab.
4. Select the target child pipeline from the dropdown (e.g., your on-prem ingestion pipeline).

## ➕ Step 3: Best Practice for Passing Parameters (Crucial)
*Note on Industry Best Practices*: When your child pipeline requires parameters (like an Array of files or JSON Objects), **do not hardcode the values directly into the Execute Pipeline activity**. If you pass an Array directly, ADF will often convert it into a String, which will break your pipeline and cost you hours of debugging.

**The correct way to bypass parameters:**
1. Click on the blank canvas of your `parent pipeline` to access the pipeline-level properties.
2. Create parameters at the **Parent Pipeline level**. Ensure the names and data types (Array, Object) perfectly match the ones in your child pipeline.
3. Click back on your **Execute Pipeline** activity.
4. Under the parameters section, click **Add dynamic content**.
5. Select the parent parameter you just created (e.g., `@pipeline().parameters.files`). This safely passes the value from the parent down to the child while retaining the strict data type.

## ➕ Step 4: Add Remaining Child Pipelines
Repeat the process to bring in the rest of your data sources.
1. Drag a second **Execute Pipeline** activity and rename it for your API workflow (e.g., `execute API`).
2. Under Settings, select your `API injection` pipeline. (If it doesn't require parameters, you don't need to configure any).
3. Drag a third **Execute Pipeline** activity and set it up for your `incremental injection` (SQL to Data Lake) pipeline.

## ➕ Step 5: Preparing for Production Alerts
Orchestration isn't just about scheduling; it's about monitoring.
- In a production scenario, if a critical pipeline fails at midnight, an on-call engineer needs to be notified immediately.
- Instead of using ADF's native metrics (which can be hard to track visually in complex flows), modern architectures use **Azure Logic Apps**.
- You will connect a Web Activity to the "Failure" path (red box) of your Execute Pipeline activities to trigger a Logic App via an API call, which will instantly dispatch an email alert containing the error details.